## Storage Strategy Benchmark

Benchmarks two partitioning approaches for the Silver tripdata table by comparing write time, storage size, number of files, and query execution time across the three required benchmark queries.

In [1]:
import os
import sys

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.common.spark_session import get_spark_session
spark = get_spark_session()
print("Project root:", PROJECT_ROOT)

26/09/13 18:32:40 WARN Utils: Your hostname, Thilinis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 172.27.138.95 instead (on interface en0)
26/09/13 18:32:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/13 18:32:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Project root: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform


## Imports, paths, and the two strategies

In [2]:
import shutil
import time
from pyspark.sql import functions as F
from deltalake import DeltaTable

SOURCE_SILVER = os.path.join(PROJECT_ROOT, "data", "delta", "silver", "tripdata")
ZONE_LOOKUP = os.path.join(PROJECT_ROOT, "data", "delta", "silver", "taxi_zone_lookup")

STRATEGY_A_PATH = os.path.join(PROJECT_ROOT, "data", "delta", "benchmark", "tripdata_by_month")
STRATEGY_B_PATH = os.path.join(PROJECT_ROOT, "data", "delta", "benchmark", "tripdata_by_day")

STRATEGIES = {
    "A: partitioned by (pickup_year, pickup_month)": (STRATEGY_A_PATH, ["pickup_year", "pickup_month"]),
    "B: partitioned by (event_date)": (STRATEGY_B_PATH, ["event_date"]),
}

## Helper functions

In [3]:
# Read the current Delta snapshot with Spark while preserving partition columns
def _read_delta(spark, path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"No data at {path} - run ingestion first")
    files = DeltaTable(path).file_uris()
    return spark.read.option("basePath", path).parquet(*files)


# # Calculate total Parquet storage size and file count, excluding the Delta log
def _dir_stats(path: str):
    total_size = 0
    file_count = 0
    for root, dirs, files in os.walk(path):
        if "_delta_log" in root:
            continue
        for f in files:
            if f.endswith(".parquet"):
                full = os.path.join(root, f)
                total_size += os.path.getsize(full)
                file_count += 1
    return total_size, file_count


def _time_it(fn):
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    return result, elapsed

## Benchmark queries

In [4]:
def query_trips_per_borough(df, zone_lookup):
    pickup_zone = zone_lookup.select(
        F.col("location_id").alias("pickup_location_id"),
        F.col("borough").alias("pickup_borough"),
    )
    return (
        df.join(F.broadcast(pickup_zone), on="pickup_location_id", how="left")
        .groupBy("pickup_borough").count()
        .collect()
    )


def query_avg_duration_per_day(df, zone_lookup):
    return (
        df.groupBy("event_date")
        .agg(F.avg("trip_duration_minutes").alias("avg_duration_minutes"))
        .collect()
    )


def query_avg_fare_per_borough(df, zone_lookup):
    pickup_zone = zone_lookup.select(
        F.col("location_id").alias("pickup_location_id"),
        F.col("borough").alias("pickup_borough"),
    )
    return (
        df.join(F.broadcast(pickup_zone), on="pickup_location_id", how="left")
        .groupBy("pickup_borough").agg(F.avg("fare_amount").alias("avg_fare"))
        .collect()
    )


QUERIES = {
    "Q1: number of taxi trips per borough": query_trips_per_borough,
    "Q2: average trip duration per day": query_avg_duration_per_day,
    "Q3: average fare per borough": query_avg_fare_per_borough,
}

## Run the benchmark

In [10]:
source_df = _read_delta(spark, SOURCE_SILVER).filter(F.col("is_valid"))
zone_lookup = _read_delta(spark, ZONE_LOOKUP)

source_row_count = source_df.count()
print(f"Source tripdata rows (is_valid=true): {source_row_count}\n")

results = {}

for label, (path, partition_cols) in STRATEGIES.items():
    print(f"{'=' * 70}\n{label}\n{'=' * 70}")

    if os.path.exists(path):
        shutil.rmtree(path)

    # ingestion time for write step only
    # load,validate and cast cost is identical regardless of partitioning
    # so only the write step actually reflects the storage strategy being compared
    from src.common.delta_writer import write_delta
    written, write_seconds = _time_it(
        lambda: write_delta(source_df, path, mode="overwrite", partition_by=partition_cols)
    )
    print(f"  ingestion time:     {write_seconds:.2f}s  ({written} rows written)")

    # storage size and file count
    size_bytes, file_count = _dir_stats(path)
    size_mb = size_bytes / (1024 * 1024)
    print(f"  storage size:   {size_mb:.2f} MB")
    print(f"  files:          {file_count}")

    # query latency
    spark.catalog.clearCache()
    strategy_df = _read_delta(spark, path).filter(F.col("is_valid"))

    query_times = {}
    for q_label, q_fn in QUERIES.items():
        spark.catalog.clearCache()
        _, elapsed = _time_it(lambda: q_fn(strategy_df, zone_lookup))
        query_times[q_label] = elapsed
        print(f"  {q_label}: {elapsed:.3f}s")

    results[label] = {
        "ingestion_time_seconds": write_seconds,
        "size_mb": size_mb,
        "file_count": file_count,
        "query_times": query_times,
    }
    print()

Source tripdata rows (is_valid=true): 9554776

A: partitioned by (pickup_year, pickup_month)


  ingestion time:     7.48s  (9554776 rows written)
  storage size:   891.95 MB
  files:          13
  Q1: number of taxi trips per borough: 0.249s
  Q2: average trip duration per day: 0.090s
  Q3: average fare per borough: 0.207s

B: partitioned by (event_date)


  ingestion time:     5.12s  (9554776 rows written)
  storage size:   847.15 MB
  files:          95
  Q1: number of taxi trips per borough: 0.114s
  Q2: average trip duration per day: 0.082s
  Q3: average fare per borough: 0.175s



## 5. Build and save the results report

In [12]:
import os

def render_report(results: dict) -> str:
    lines = [
        "# Task 6 — Storage Strategy Benchmark",
        "",
        "Two partitioning strategies for the `tripdata` silver table, benchmarked "
        "on ingestion (write) time, storage size, file count, and query latency "
        "for the three required queries.",
        "",
        "| Metric | " + " | ".join(results.keys()) + " |",
        "|---|" + "---|" * len(results),
        "| Write time (s) | " + " | ".join(f"{r['ingestion_time_seconds']:.2f}" for r in results.values()) + " |",
        "| Storage size (MB) | " + " | ".join(f"{r['size_mb']:.2f}" for r in results.values()) + " |",
        "| Number of files | " + " | ".join(str(r["file_count"]) for r in results.values()) + " |",
    ]
    query_labels = list(next(iter(results.values()))["query_times"].keys())
    for q in query_labels:
        row = [f"| {q} (s)"] + [f"{r['query_times'][q]:.3f}" for r in results.values()]
        lines.append(" | ".join(row) + " |")
    lines.append("")
    return "\n".join(lines)


report_text = render_report(results)

out_dir = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "benchmark_results.md")
with open(out_path, "w") as f:
    f.write(report_text)

print(f"Saved: {out_path}")

Saved: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/reports/benchmark_results.md


## 6. Render the results table inline

In [13]:
from IPython.display import Markdown, display
display(Markdown(report_text))

# Task 6 — Storage Strategy Benchmark

Two partitioning strategies for the `tripdata` silver table, benchmarked on ingestion (write) time, storage size, file count, and query latency for the three required queries.

| Metric | A: partitioned by (pickup_year, pickup_month) | B: partitioned by (event_date) |
|---|---|---|
| Write time (s) | 7.48 | 5.12 |
| Storage size (MB) | 891.95 | 847.15 |
| Number of files | 13 | 95 |
| Q1: number of taxi trips per borough (s) | 0.249 | 0.114 |
| Q2: average trip duration per day (s) | 0.090 | 0.082 |
| Q3: average fare per borough (s) | 0.207 | 0.175 |
